In [ ]:
# ######################## DirectDensity-Conservation PINN ################################
# Direct-density formulation with global ion conservation
#########################################################################

import os
import time
import gc
import random

import numpy as np
import pandas as pd
import tensorflow as tf

# ----------------------------------------------------------------------
# Working directory
# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# Execution switches
# ----------------------------------------------------------------------
# For a completely new run, set all three switches to True.
RUN_PREBIAS = True
RUN_FD = True
RUN_COMPARE = True

# ----------------------------------------------------------------------
# Ablation-case information
# ----------------------------------------------------------------------
CASE_NAME = "ablation_direct_cons"

FORMULATION = "Direct density with conservation"

DENSITY_REPRESENTATION = "direct"

USE_CONSERVATION = True

BASE_WCONS = 100.0

# ----------------------------------------------------------------------
# Representative operating point
# ----------------------------------------------------------------------
BASE_P0 = 1.0e21
BASE_PHI = 0.10

# ----------------------------------------------------------------------
# Network architecture
# ----------------------------------------------------------------------
BASE_ACTIVATION = "tanh"
BASE_HIDDEN_LAYERS = 4
BASE_NEURONS = 40

# ----------------------------------------------------------------------
# Training settings
# ----------------------------------------------------------------------
BASE_NUM_EPOCH_ADAM = 2000
BASE_NUM_EPOCH_LBFGS = 10000
BASE_RUN_LBFGS = True

# ----------------------------------------------------------------------
# Independent random seeds
# ----------------------------------------------------------------------
SEEDS = [42, 1234, 2024]

# ----------------------------------------------------------------------
# Output directories
# ----------------------------------------------------------------------
OUTPUT_ROOT = "results"
BASE_DIR = os.path.join(OUTPUT_ROOT, CASE_NAME)

os.makedirs(BASE_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# Summary files
# ----------------------------------------------------------------------
summary_path = os.path.join(
    BASE_DIR,
    "ablation_direct_cons_per_seed_summary.csv",
)

agg_path = os.path.join(
    BASE_DIR,
    "ablation_direct_cons_mean_std_summary.csv",
)

# ----------------------------------------------------------------------
# Load an existing summary if available
# ----------------------------------------------------------------------
if os.path.exists(summary_path):
    rows = pd.read_csv(summary_path).to_dict("records")
else:
    rows = []

# ----------------------------------------------------------------------
# Objects that may remain in memory after a previous notebook run
# ----------------------------------------------------------------------
LEFTOVERS = [
    "prebias_model",
    "main_model",
    "model",
    "layers",
    "results",
    "X_0",
    "X_r",
    "X_a",
    "X_c",
    "u_a",
    "u_c",
    "X_r_pre",
    "X_0_pre",
    "X_aPre",
    "X_cPre",
    "lb",
    "ub",
    "lb_pre",
    "ub_pre",
    "lb_main",
    "ub_main",
    "u0_interp_dict",
    "u_0",
    "x_0",
    "X_test",
    "X_list",
    "T_list",
    "psi_pred",
    "n_pred",
    "I_pred",
]

# ----------------------------------------------------------------------
# Helper functions
# ----------------------------------------------------------------------
def require_files(outdir, filenames):
    """
    Check whether all files required for PINN–FD comparison exist.
    """
    missing = [
        filename
        for filename in filenames
        if not os.path.exists(os.path.join(outdir, filename))
    ]

    if missing:
        raise FileNotFoundError(
            f"Missing files in {outdir}: {missing}"
        )

def set_all_seeds(seed):
    """
    Reset the TensorFlow session and set all random seeds.
    """
    tf.keras.backend.clear_session()
    gc.collect()

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def remove_existing_case(rows_list, seed):
    """
    Remove an existing result for the same baseline case and seed.

    This prevents duplicate rows when a case is rerun.
    """
    filtered_rows = []

    for row in rows_list:

        same_case = row.get("case_name", CASE_NAME) == CASE_NAME

        try:
            same_P0 = np.isclose(
                float(row.get("P0", np.nan)),
                float(BASE_P0),
            )
        except (TypeError, ValueError):
            same_P0 = False

        try:
            same_seed = int(row.get("seed", -1)) == int(seed)
        except (TypeError, ValueError):
            same_seed = False

        if not (same_case and same_P0 and same_seed):
            filtered_rows.append(row)

    return filtered_rows

# ======================================================================
# Run the direct-density formulation with conservation
# ======================================================================
for seed in SEEDS:

    tag = (
        f"P0_{BASE_P0:.0e}_"
        f"phi{BASE_PHI:.2f}_"
        f"{BASE_ACTIVATION}_"
        f"{BASE_HIDDEN_LAYERS}x{BASE_NEURONS}_"
        f"seed{seed}"
    )

    OUTPUT_DIR = os.path.join(
        BASE_DIR,
        tag,
    )

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True,
    )

    print("\n" + "=" * 90)
    print(f"Ablation case     : {CASE_NAME}")
    print(f"Formulation       : {FORMULATION}")
    print(f"Density variable  : {DENSITY_REPRESENTATION}")
    print(f"Conservation      : {USE_CONSERVATION}")
    print(f"Tag               : {tag}")
    print(f"P0                : {BASE_P0:.3e} m^-3")
    print(f"phi_a = phi_c     : {BASE_PHI:.2f} eV")
    print(f"Activation        : {BASE_ACTIVATION}")
    print(
        f"Architecture      : "
        f"{BASE_HIDDEN_LAYERS} hidden layers × "
        f"{BASE_NEURONS} neurons"
    )
    print(f"W_cons            : {BASE_WCONS}")
    print(f"Adam epochs       : {BASE_NUM_EPOCH_ADAM}")
    print(f"L-BFGS iterations : {BASE_NUM_EPOCH_LBFGS}")
    print(f"Run L-BFGS        : {BASE_RUN_LBFGS}")
    print(f"Seed              : {seed}")
    print(f"Output directory  : {OUTPUT_DIR}")
    print("=" * 90)

    # ------------------------------------------------------------------
    # Reset session and random seeds
    # ------------------------------------------------------------------
    set_all_seeds(seed)

    g = globals()

    # Remove objects left from a previous seed.
    for name in LEFTOVERS:
        if name in g:
            del g[name]

    # ------------------------------------------------------------------
    # Pass parameters to the PINN and FD notebooks
    # ------------------------------------------------------------------
    g["P0"] = float(BASE_P0)

    g["phia"] = float(BASE_PHI)
    g["phic"] = float(BASE_PHI)

    g["OUTPUT_DIR"] = OUTPUT_DIR

    g["ACTIVATION"] = BASE_ACTIVATION
    g["HIDDEN_LAYERS"] = int(BASE_HIDDEN_LAYERS)
    g["NEURONS"] = int(BASE_NEURONS)

    g["W_cons"] = float(BASE_WCONS)

    g["NUM_EPOCH_ADAM"] = int(BASE_NUM_EPOCH_ADAM)
    g["NUM_EPOCH_LBFGS"] = int(BASE_NUM_EPOCH_LBFGS)
    g["RUN_LBFGS"] = bool(BASE_RUN_LBFGS)

    g["SEED"] = int(seed)

    t_case0 = time.time()

    try:

        # --------------------------------------------------------------
        # Run the steady-state pre-bias PINN
        # --------------------------------------------------------------
        if RUN_PREBIAS:

            t0 = time.time()

            get_ipython().run_line_magic("run", '-i "CA_PINN_Solver.ipynb"')
            pinn_runtime = time.time() - t0

            print(
                f"[{tag}] Pre-bias PINN runtime: "
                f"{pinn_runtime:.1f} s"
            )

        else:
            pinn_runtime = np.nan

        # --------------------------------------------------------------
        # Run the finite-difference reference model
        # --------------------------------------------------------------
        if RUN_FD:

            t0 = time.time()

            get_ipython().run_line_magic("run", '-i "FD_Solver.ipynb"')
            fd_runtime = time.time() - t0

            print(
                f"[{tag}] FD runtime: "
                f"{fd_runtime:.1f} s"
            )

        else:
            fd_runtime = np.nan

        # --------------------------------------------------------------
        # Compare PINN and FD solutions
        # --------------------------------------------------------------
        if RUN_COMPARE:

            require_files(
                OUTPUT_DIR,
                [
                    "pinn_psi.csv",
                    "pinn_n.csv",
                    "pinn_I.csv",
                    "fd_psi.csv",
                    "fd_n.csv",
                    "fd_I.csv",
                ],
            )

            from validation_utils import compare_and_plot

            metrics = compare_and_plot(
                title_suffix=tag,
                base_dir=OUTPUT_DIR,
            )

            # ----------------------------------------------------------
            # Positivity diagnostics
            # ----------------------------------------------------------
            pinn_n_df = pd.read_csv(
                os.path.join(
                    OUTPUT_DIR,
                    "pinn_n.csv",
                )
            )

            pinn_I_df = pd.read_csv(
                os.path.join(
                    OUTPUT_DIR,
                    "pinn_I.csv",
                )
            )

            min_n = float(
                pinn_n_df["n_m3"].min()
            )

            min_I = float(
                pinn_I_df["I_m3"].min()
            )

            print(
                f"Minimum predicted electron density: "
                f"{min_n:.3e} m^-3"
            )

            print(
                f"Minimum predicted ion density: "
                f"{min_I:.3e} m^-3"
            )

            case_runtime = time.time() - t_case0

            # Remove an older result for the same seed.
            rows = remove_existing_case(
                rows,
                seed=seed,
            )

            rows.append(
                {
                    "case_name": CASE_NAME,
                    "formulation": FORMULATION,
                    "density_representation": DENSITY_REPRESENTATION,
                    "conservation": USE_CONSERVATION,

                    "P0": float(BASE_P0),
                    "phi": float(BASE_PHI),

                    "activation": BASE_ACTIVATION,
                    "hidden_layers": int(BASE_HIDDEN_LAYERS),
                    "neurons": int(BASE_NEURONS),

                    "num_epoch_adam": int(
                        BASE_NUM_EPOCH_ADAM
                    ),

                    "num_epoch_lbfgs": int(
                        BASE_NUM_EPOCH_LBFGS
                    ),

                    "run_lbfgs": bool(
                        BASE_RUN_LBFGS
                    ),

                    "W_cons": float(BASE_WCONS),

                    "seed": int(seed),

                    "tag": tag,
                    "output_dir": OUTPUT_DIR,

                    "relL2_psi": float(
                        metrics["relL2_psi"]
                    ),

                    "relL2_n": float(
                        metrics["relL2_n"]
                    ),

                    "relL2_I": float(
                        metrics["relL2_I"]
                    ),

                    "Imean_PINN": float(
                        metrics["I_mean_PINN"]
                    ),

                    "Imean_FD": float(
                        metrics["I_mean_FD"]
                    ),

                    "Imean_rel_error": float(
                        metrics["I_mean_rel_error"]
                    ),

                    "min_n": min_n,
                    "min_I": min_I,

                    "pinn_runtime_s": float(pinn_runtime)
                    if np.isfinite(pinn_runtime)
                    else np.nan,

                    "fd_runtime_s": float(fd_runtime)
                    if np.isfinite(fd_runtime)
                    else np.nan,

                    "case_runtime_s": float(case_runtime),

                    "status": "ok",
                }
            )

            pd.DataFrame(rows).to_csv(
                summary_path,
                index=False,
            )

            print(
                f"[{tag}] Updated per-seed summary:\n"
                f"{summary_path}"
            )

    except Exception as e:

        case_runtime = time.time() - t_case0

        print(
            f"[{tag}] FAILED: {repr(e)}"
        )

        # Remove an older result for the same seed.
        rows = remove_existing_case(
            rows,
            seed=seed,
        )

        rows.append(
            {
                "case_name": CASE_NAME,
                "formulation": FORMULATION,
                "density_representation": DENSITY_REPRESENTATION,
                "conservation": USE_CONSERVATION,

                "P0": float(BASE_P0),
                "phi": float(BASE_PHI),

                "activation": BASE_ACTIVATION,
                "hidden_layers": int(BASE_HIDDEN_LAYERS),
                "neurons": int(BASE_NEURONS),

                "num_epoch_adam": int(
                    BASE_NUM_EPOCH_ADAM
                ),

                "num_epoch_lbfgs": int(
                    BASE_NUM_EPOCH_LBFGS
                ),

                "run_lbfgs": bool(
                    BASE_RUN_LBFGS
                ),

                "W_cons": float(BASE_WCONS),

                "seed": int(seed),

                "tag": tag,
                "output_dir": OUTPUT_DIR,

                "relL2_psi": np.nan,
                "relL2_n": np.nan,
                "relL2_I": np.nan,

                "Imean_PINN": np.nan,
                "Imean_FD": np.nan,
                "Imean_rel_error": np.nan,

                "min_n": np.nan,
                "min_I": np.nan,

                "pinn_runtime_s": np.nan,
                "fd_runtime_s": np.nan,

                "case_runtime_s": float(case_runtime),

                "status": f"failed: {type(e).__name__}",
            }
        )

        pd.DataFrame(rows).to_csv(
            summary_path,
            index=False,
        )

        continue

# ======================================================================
# Aggregate the successful seed results
# ======================================================================
df = pd.DataFrame(rows)

df.to_csv(
    summary_path,
    index=False,
)

if df.empty:

    print("\nNo results were generated.")

else:

    ok = df[
        (df["status"] == "ok")
        & (df["case_name"] == CASE_NAME)
        & np.isclose(
            df["P0"].astype(float),
            BASE_P0,
        )
    ].copy()

    metric_cols = [
        "relL2_psi",
        "relL2_n",
        "relL2_I",
        "Imean_PINN",
        "Imean_FD",
        "Imean_rel_error",
        "min_n",
        "min_I",
        "pinn_runtime_s",
        "fd_runtime_s",
        "case_runtime_s",
    ]

    agg_rows = []

    for metric in metric_cols:

        metric_values = pd.to_numeric(
            ok[metric],
            errors="coerce",
        ).dropna()

        if len(metric_values) > 0:

            metric_mean = float(
                metric_values.mean()
            )

            metric_std = (
                float(metric_values.std(ddof=1))
                if len(metric_values) > 1
                else np.nan
            )

        else:

            metric_mean = np.nan
            metric_std = np.nan

        agg_rows.append(
            {
                "case_name": CASE_NAME,
                "formulation": FORMULATION,
                "density_representation": DENSITY_REPRESENTATION,
                "conservation": USE_CONSERVATION,

                "P0": float(BASE_P0),
                "phi": float(BASE_PHI),

                "activation": BASE_ACTIVATION,
                "hidden_layers": int(BASE_HIDDEN_LAYERS),
                "neurons": int(BASE_NEURONS),

                "W_cons": float(BASE_WCONS),

                "metric": metric,
                "mean": metric_mean,
                "std": metric_std,

                "n_runs": int(
                    len(metric_values)
                ),
            }
        )

    agg = pd.DataFrame(
        agg_rows
    )

    agg.to_csv(
        agg_path,
        index=False,
    )

    print("\n" + "=" * 90)
    print("DIRECT-DENSITY + CONSERVATION ABLATION CASE COMPLETED")
    print("=" * 90)

    print(
        f"Per-seed summary:\n"
        f"{summary_path}"
    )

    print(
        f"\nMean ± standard-deviation summary:\n"
        f"{agg_path}"
    )

    print(
        f"\nSuccessful runs: "
        f"{len(ok)}/{len(SEEDS)}"
    )

    print("\nAggregated results:")
    print(
        agg.to_string(index=False)
    )